In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### Ground Truth & Model Output

In [2]:
ground_truths = [
    {
        "image_id": "image_1",
        "boxes": np.array([
            [10, 10, 50, 50],
            [60, 60, 100, 100]
        ]),
        "labels": np.array([1, 2])
    },
    {
        "image_id": "image_2",
        "boxes": np.array([
            [15, 15, 55, 55]
        ]),
        "labels": np.array([1])
    },
    {
        "image_id": "image_3",
        "boxes": np.array([
            [30, 30, 70, 70]
        ]),
        "labels": np.array([2])
    }
]

predictions = [
    {
        "image_id": "image_1",
        "boxes": np.array([
            [11, 11, 49, 49],
            [62, 62, 98, 98],
            [0, 0, 20, 20],
            [12, 12, 48, 48]
        ]),
        "labels": np.array([1, 2, 1, 1]),
        "scores": np.array([0.95, 0.72, 0.65, 0.55])
    },
    {
        "image_id": "image_2",
        "boxes": np.array([
            [16, 16, 54, 54],
            [70, 70, 100, 100]
        ]),
        "labels": np.array([1, 2]),
        "scores": np.array([0.60, 0.40])
    },
    {
        "image_id": "image_3",
        "boxes": np.array([
            [31, 31, 69, 69],
            [0, 0, 15, 15]
        ]),
        "labels": np.array([2, 1]),
        "scores": np.array([0.48, 0.35])
    }
]

print("Ground-truth images:", len(ground_truths))
print("Prediction images:", len(predictions))

Ground-truth images: 3
Prediction images: 3


#### IoU Calculation

In [3]:
def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_width = max(0, x2 - x1)
    intersection_height = max(0, y2 - y1)
    intersection_area = intersection_width * intersection_height

    box1_area = max(0, box1[2] - box1[0]) * max(
        0, box1[3] - box1[1]
    )
    box2_area = max(0, box2[2] - box2[0]) * max(
        0, box2[3] - box2[1]
    )

    union_area = box1_area + box2_area - intersection_area
    if union_area == 0:
        return 0.0

    return intersection_area / union_area

#### Threshold Evaluation Function

In [4]:
def evaluate_threshold(
    predictions,
    ground_truths,
    confidence_threshold,
    iou_threshold=0.50
):
    prediction_map = {item["image_id"]: item for item in predictions}
    ground_truth_map = {
        item["image_id"]: item for item in ground_truths
    }

    image_ids = set(prediction_map) | set(ground_truth_map)
    true_positive = 0
    false_positive = 0
    false_negative = 0

    for image_id in image_ids:
        prediction = prediction_map.get(
            image_id,
            {
                "boxes": np.empty((0, 4)),
                "labels": np.array([]),
                "scores": np.array([])
            }
        )

        ground_truth = ground_truth_map.get(
            image_id,
            {
                "boxes": np.empty((0, 4)),
                "labels": np.array([])
            }
        )

        prediction_order = np.argsort(-prediction["scores"])
        matched_ground_truths = set()
        
        for prediction_index in prediction_order:
            score = prediction["scores"][prediction_index]
            if score < confidence_threshold:
                continue

            predicted_box = prediction["boxes"][prediction_index]
            predicted_label = prediction["labels"][prediction_index]

            best_iou = 0.0
            best_ground_truth_index = -1

            for ground_truth_index, ground_truth_box in enumerate(
                ground_truth["boxes"]
            ):
                if ground_truth_index in matched_ground_truths:
                    continue

                ground_truth_label = ground_truth["labels"][
                    ground_truth_index
                ]

                if predicted_label != ground_truth_label:
                    continue

                current_iou = calculate_iou(
                    predicted_box,
                    ground_truth_box
                )

                if current_iou > best_iou:
                    best_iou = current_iou
                    best_ground_truth_index = ground_truth_index

            if (
                best_iou >= iou_threshold
                and best_ground_truth_index != -1
            ):
                true_positive += 1
                matched_ground_truths.add(
                    best_ground_truth_index
                )
            else:
                false_positive += 1

        false_negative += (
            len(ground_truth["boxes"])
            - len(matched_ground_truths)
        )

    precision = true_positive / max(
        true_positive + false_positive, 1
    )

    recall = true_positive / max(true_positive + false_negative, 1)

    f1_score = (
        2 * precision * recall / max(precision + recall, 1e-12)
    )

    return {
        "threshold": confidence_threshold,
        "TP": true_positive,
        "FP": false_positive,
        "FN": false_negative,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score
    }